# 01_jsonrpc_tools

Corresponds to `index.md` stage 1: a teaching subset of MCP over stdio.

Notebook path: `codes/mcp/01_jsonrpc_tools/qa.ipynb`. File I/O for this stage
stays in this directory (`server.py`, `client.py`).

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`client (JSON-RPC) → stdio (NDJSON) → server.handle → Tool.handler → content[]`

Real MCP uses Content-Length framing; this lab uses one JSON object per line so
every message is printable. Method names (`initialize`, `tools/list`,
`tools/call`) match the stable tools loop.


In [ ]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_jsonrpc_tools" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "01_jsonrpc_tools"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/01_jsonrpc_tools")

ROOT = lab_root()
print(ROOT)


## 1. Read a `tools/list` result

`payload` is what a server might return for `tools/list`. Print every tool
`name`, then print the `required` fields of the tool named `add`.


In [ ]:
payload = {
    "tools": [
        {
            "name": "echo",
            "description": "Return the given text unchanged.",
            "inputSchema": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
        {
            "name": "add",
            "description": "Add two numbers and return the sum as text.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                },
                "required": ["a", "b"],
            },
        },
    ]
}

# answer

names = [tool["name"] for tool in payload["tools"]]
print("names:", names)
add = next(tool for tool in payload["tools"] if tool["name"] == "add")
print("add.required:", add["inputSchema"]["required"])


## 2. Dispatch `tools/call` by hand

`handle_call(tools, name, arguments)` should:

1. Find the tool by `name`
2. Call `tool(arguments)` when found and return
   `{"content": [{"type": "text", "text": ...}], "isError": False}`
3. Return `{"content": [{"type": "text", "text": "unknown tool"}], "isError": True}`
   when the name is missing

Print both the happy path for `add` and the unknown-tool path.


In [ ]:
def add(arguments: dict) -> str:
    return str(float(arguments["a"]) + float(arguments["b"]))

tools = {"add": add}

# answer

def handle_call(tools: dict, name: str, arguments: dict) -> dict:
    tool = tools.get(name)
    if tool is None:
        return {
            "content": [{"type": "text", "text": "unknown tool"}],
            "isError": True,
        }
    text = tool(arguments)
    return {
        "content": [{"type": "text", "text": text}],
        "isError": False,
    }

print(handle_call(tools, "add", {"a": 2, "b": 3}))
print(handle_call(tools, "mul", {"a": 2, "b": 3}))


## 3. Drive the real lab server

Import `StdioClient` from `client.py` in this directory. Then:

1. `initialize`
2. notify `notifications/initialized`
3. `tools/list` and print tool names
4. `tools/call` on `echo` with `{"text": "ping"}` and print the text content

Close the client in a `finally` block.


In [ ]:
import sys

sys.path.insert(0, str(ROOT))
from client import StdioClient

# answer

client = StdioClient(ROOT / "server.py")
try:
    init = client.request(
        "initialize",
        {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "qa", "version": "0.1.0"},
        },
    )
    print("server:", init["result"]["serverInfo"]["name"])
    client.notify("notifications/initialized")
    listed = client.request("tools/list")
    print("tools:", [t["name"] for t in listed["result"]["tools"]])
    called = client.request("tools/call", {"name": "echo", "arguments": {"text": "ping"}})
    print("echo:", called["result"]["content"][0]["text"])
finally:
    client.close()


## 4. Register a third tool

Open `server.py` conceptually: tools live in the `TOOLS` dict. In this cell,
build a local `TOOLS`-like mapping that also has `mul(a, b)`, then use the
`handle_call` from question 2 (or rewrite it) to call `mul` with `a=4, b=5`.
Print the returned text (`"20.0"` or `"20"` are both fine).

This is the extension point later stages keep: handlers stay ordinary Python
callables; only the transport changes.


In [ ]:
# answer

def mul(arguments: dict) -> str:
    return str(float(arguments["a"]) * float(arguments["b"]))

local_tools = {"add": add, "mul": mul}
result = handle_call(local_tools, "mul", {"a": 4, "b": 5})
print(result["content"][0]["text"])
